# 2. データを分析する

**どんな人が生き残ったのか**を、答えの付いている学習用データから調べる。

ここで「効きそうな項目」に見当を付けておくと、この後の作業で
どの項目を大事に扱うかを決められる。

In [ ]:
import pandas as pd

# 分析に使うのは学習用データだけ。test には survived が無いので、
# 「どんな人が生き残ったか」は train からしか分からない
train = pd.read_csv("../data/raw/train.csv", index_col=0)

## 1. そもそも何人が生き残ったのか

`survived` は、助かった人が `1`、助からなかった人が `0` で入っている。
まずそれぞれ何人いるのかを数える。

In [ ]:
# .value_counts() は、その項目に出てくる値ごとに人数を数える
train["survived"].value_counts()

# 出力の見方
#   0 が 266 人（助からなかった）、1 が 179 人（助かった）
#   Name: count は「これは人数の列です」という pandas が付けたラベル。データではない

In [ ]:
# normalize=True を付けると、人数ではなく割合になる
train["survived"].value_counts(normalize=True)

# 助かったのは約 40%
#
# この 40% が後で重要になる。何も考えずに「全員助からなかった」と答えるだけでも、
# 60% は当たってしまう。つまり正解率 60% は「何も学べていない」のと同じ。
# こういう最低ラインのことを ベースライン と呼ぶ

## 2. 数値の項目をまとめて眺める

`describe()` を使うと、数字の項目について「だいたいどのあたりの値なのか」が一度に分かる。
極端に大きい値が混ざっていないかを見つけるのにも使う。

In [ ]:
# .describe() は数字の項目だけを対象に、代表的な数値をまとめて出す
#   文字の項目（sex, embarked）は自動で除かれる
train.describe()

# 出力の見方（縦に並ぶ行の意味）
#   count  値が入っている人数。age だけ 360 と少ないのは、空いている人がいるため
#   mean   平均値
#   std    ばらつきの大きさ。値が散らばっているほど大きくなる
#   min    最小値
#   25%    小さい順に並べて、4 分の 1 番目の人の値
#   50%    ちょうど真ん中の人の値
#   75%    小さい順に並べて、4 分の 3 番目の人の値
#   max    最大値

### 読み取れること

**`pclass` の 50%（真ん中の人）が 3** — 半数以上が一番下の3等客室に乗っていた。

**`age` の平均は約 29 歳** — 若い人が多い。最小は 0.67（生後8か月）、最大は 80 歳。

**`survived` の平均が 0.402** — `0` と `1` だけの項目は、平均がそのまま「`1` の割合」になる。
つまり 40.2% が生還。この性質は、後で属性ごとの生存率を出すときに使う。

**`fare`（運賃）が特殊** — 平均 34 に対して最大 512。平均の15倍を払った人がいる。
この項目は次で詳しく見る。

### 平均・中央値・最頻値

代表的な数値には3種類あり、それぞれ英語名がある。この先ずっと出てくる。

| 日本語 | 英語 | pandas | どういう数字か |
| --- | --- | --- | --- |
| 平均値 | **mean** | `.mean()` | 全部足して人数で割る |
| 中央値 | **median** | `.median()` | 順に並べてちょうど真ん中の人の値 |
| 最頻値 | **mode** | `.mode()` | 一番多く出てくる値 |

`describe()` の `50%` の行が中央値にあたる。

**平均と中央値は、大きくずれることがある。**

| 項目 | 平均 | 中央値 | 平均より下の人 |
| --- | --- | --- | --- |
| `age` | 29.21 | 28.00 | 53.3% |
| `fare` | 33.96 | **15.00** | **75.1%** |

`age` はほぼ同じだが、`fare` は倍以上ひらく。512 を払った人のような一部の高額客が
平均を引き上げるためで、結果として**4人に3人が「平均以下」**になってしまう。

これだと平均は代表値として役に立たない。テレビで年収の話をするときに
「平均」と「中央値」でもめるのと同じ理由で、
一部の人が極端に高い項目では中央値の方が実感に近くなる。

最頻値は、`embarked`（港）のように**大小を比べられない値**にも使えるのが強み。
港に平均や真ん中はないが、「一番多いのはどこか」なら答えられる。

## 3. どの項目が生死と関係していそうか

項目を1つずつ調べる前に、**相関係数**を使ってまとめて絞り込む。

相関係数は、2つの項目が**どれくらい連動しているか**を -1 〜 1 の数字で表したもの。

| 値 | 意味 |
| --- | --- |
| **1 に近い** | 片方が大きいとき、もう片方も大きい |
| **-1 に近い** | 片方が大きいとき、もう片方は小さい |
| **0 に近い** | 連動していない |

ただし相関係数は**数字どうしでしか計算できない**。
`sex` は `male` / `female` という文字なので、先に数字へ変換する。

In [ ]:
# get_dummies() は、文字の項目を 0/1 の項目に置き換える
#   この操作を ダミー化（One-Hot エンコーディング）と呼ぶ
#
#   文字の項目すべてが対象なので、sex と embarked がまとめて変換される
#     sex      → sex_female / sex_male                （2 種類なので 2 項目）
#     embarked → embarked_C / embarked_Q / embarked_S  （3 種類なので 3 項目）
#   自分に当てはまる項目だけが True（はい）になり、残りは False（いいえ）になる
#
#   変換前              変換後
#   sex     embarked    sex_female  sex_male  embarked_C  embarked_Q  embarked_S
#   female  S      →    True        False     False       False       True
#   male    C      →    False       True      True        False       False
pd.get_dummies(train).head()

In [ ]:
# .corrwith() は、表の各列と、指定した1つの列との相関係数をまとめて出す
pd.get_dummies(train).corrwith(train["survived"])

# 出力の見方
#   survived が 1.000 なのは自分自身との相関なので、当然そうなる。判断には使わない

### 読み取れること

| 項目 | 相関係数 | 意味 |
| --- | --- | --- |
| `sex_female` | **+0.559** | 女性ほど助かっている。最も強い |
| `sex_male` | **-0.559** | 男性ほど助かっていない |
| `pclass` | **-0.358** | 数字が大きい（下の等級）ほど助かっていない |
| `fare` | +0.259 | 運賃が高いほど助かっている |
| `embarked_C` | +0.183 | シェルブール港から乗った人はやや多く助かっている |
| `embarked_S` | -0.173 | サウサンプトン港から乗った人はやや少ない |
| `age` | -0.081 | ほとんど関係が見えない |
| `sibsp` | -0.045 | ほとんど関係が見えない |
| `parch` | +0.080 | ほとんど関係が見えない |

**効いているのは性別と客室の等級。** 次が運賃だが、良い客室ほど運賃は高いので、
等級と同じことを別の角度から見ているだけかもしれない。

読むときに注意したい点が3つある。

**1. `sex_female` と `sex_male` は同じことを言っている**

男でなければ女なので、片方が決まればもう片方も決まる。
符号が逆なだけで中身は同じ。学習には片方あれば足りる。

**2. `pclass` がマイナスなのは「下の等級ほど助からなかった」という意味**

`pclass` は 1 が最上級で 3 が最下級。**数字の大小と豪華さが逆向き**になっている。
だから「数字が大きいほど `survived` が小さい」＝「下の等級ほど助からなかった」となる。
符号だけ見て「マイナスだから悪い」と判断しないこと。

**3. `age` の相関が弱くても「年齢は関係ない」とは言えない**

相関係数が見ているのは「**一方向に増えたら他方も増える**」という関係だけ。

タイタニックでは子どもが優先されたと言われている。もしそうなら
「子どもは助かり、大人は助からず、高齢者はまた助かる」という**山なりの関係**になるが、
これは一方向の増減ではないので、相関係数では 0 に近く出てしまう。

実際、後の章でモデルを作ると、年齢は3番目に効いている項目として現れる。

## 4. 実際の生存率を属性ごとに確かめる

相関係数はあくまで目安。効いていそうな項目について、実際の生存率を出して裏を取る。

`survived` は `0` と `1` なので、**グループごとの平均がそのグループの生存率**になる。
（2 で見た「0/1 の項目の平均は 1 の割合になる」という性質を使う）

In [ ]:
# .groupby("列名") でその列の値ごとにグループ分けし、.mean() で各グループの平均を出す
#   ["survived"] で対象の列を絞ってから平均を取る
train.groupby("pclass")["survived"].mean()

# 出力の見方
#   1（1等客室）: 0.685 → 68.5% が生還
#   2（2等客室）: 0.443 → 44.3%
#   3（3等客室）: 0.258 → 25.8%

In [ ]:
train.groupby("sex")["survived"].mean()

# female: 0.776 → 77.6% が生還
# male  : 0.201 → 20.1%

In [ ]:
# 生存率だけ見ると人数を見落とすので、.agg() で複数の集計を同時に出しておく
#   count は人数。「9割生還」でも 2 人中 2 人なら、たまたまかもしれない
train.groupby("sex")["survived"].agg(["mean", "count"])

# female 156 人 / male 289 人。どちらも十分な人数があるので、この差は偶然ではなさそう

In [ ]:
# 性別と客室クラスを掛け合わせて見る。両方を同時に指定するとグループが細かくなる
train.groupby(["sex", "pclass"])["survived"].agg(["mean", "count"])

# 3等客室の女性（58.5%）は、1等客室の男性（43.6%）より生存率が高い。
# 客室クラスより性別の影響の方が大きかったことが、ここからも読み取れる
#
# 女性は 1等 94.3% → 2等 86.8% → 3等 58.5%
# 男性は 1等 43.6% → 2等 16.9% → 3等 13.7%
# どの等級でも女性が男性を上回っている

## この章のまとめ

- 学習用データ 445 人のうち生還は 179 人、**生存率は 40.2%**。
  何も学習していないモデルでも 60% 当たるので、精度はこれを上回って初めて意味がある
- **性別が最も効く。** 女性 77.6% に対して男性 20.1%
- **次に客室クラス。** 1等 68.5% → 2等 44.3% → 3等 25.8%
- 性別と客室クラスを掛け合わせると、**3等の女性（58.5%）が1等の男性（43.6%）を上回る**。
  どの等級でも女性が男性より高く、性別の影響の方が強い
- `age` は相関係数上は弱いが、区切り方次第で効く可能性が残っている